> **Purpose:** STDR recursive partition: first-layer Fiedler, bipartition Jaccard, ARI.  
> **Data:** synthesized

# Figure 10 — Final-partition recovery under sub-sampling

**Motivation.** The first-layer Fiedler bipartition is a single cut of the $n$ taxa.
STDR’s partition phase recursively re-applies Fiedler on each subset until cluster
size $\leq$ threshold, producing a binary tree over the leaves. Sub-sampling can
preserve the *first* cut while corrupting *deeper* splits. We report two complementary
recovery metrics for each $(n, p, \text{seed})$:

1. **First-layer sign-recovery** $\max(s, 1 - s)$ where $s$ is the sign agreement
   between Fiedler$(L(M))$ and Fiedler$(L(\hat S))$. Reaches 1 as soon as the top
   bipartition is recovered.
2. **Bipartition Jaccard** $|B_M \cap B_{\hat S}| / |B_M \cup B_{\hat S}|$ over
   the *set* of bipartitions emitted by the recursion (each internal split, in
   full-taxon indexing, canonicalised to the smaller side). Robinson-Foulds-style:
   directly counts preserved tree edges. Headline metric.
3. **Final-partition ARI** Adjusted Rand Index on the leaf-cluster assignments.
   Flat-clustering view; reported for comparison — ARI adjusts for chance against
   *random labellings* which is the wrong null when partial hierarchical recovery
   has occurred.

Each trial sub-samples $S$ once; we report per-trial means across seeds (no
$\hat S$ averaging — the production sweep in  does Welford
averaging across $K$ reps; here we follow the figure-1 single-shot pattern).


## Cell 1 — Imports & configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "setup.py").exists():
    ROOT = ROOT.parent
PROJECT_ROOT = ROOT / "sub_sampled_fielder_vec"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.utils import (
    build_balanced_binary_S, compute_fiedler_of_S, subsample_S,
    compute_recovery,
    make_full_key, get_or_compute_full,
)
from src.utils.recursive_partition import recursive_partition
from src.utils.metrics import compute_final_partition_agreement


ALPHA       = 0.9
K_VALUES    = [6, 7, 8, 9, 10, 11]       # n = 64, 128, 256, 512, 1024, 2048
N_VALUES    = [2 ** k for k in K_VALUES]
P_GRID      = np.geomspace(1e-3, 1.0, 20)
N_TRIALS    = 10
THRESHOLD   = 8                          # recursion stop: |I| <= THRESHOLD
NUM_GAPS    = 0
MIN_SPLIT   = 1
CACHE_DIR   = PROJECT_ROOT / "cache" / "theoretical_interpretation"

# Smoke-test override — uncomment for a fast end-to-end pass.
# K_VALUES, P_GRID, N_TRIALS = [5, 6], np.geomspace(1e-2, 1.0, 8), 3
# N_VALUES = [2 ** k for k in K_VALUES]

print(f"alpha={ALPHA}, n_values={N_VALUES}, threshold={THRESHOLD}")
print(f"|p_grid|={len(P_GRID)}, n_trials={N_TRIALS}")
print(f"cache_dir={CACHE_DIR}")


## Cell 2 — Build $S$ and reference partitions per $n$
Per $n$: closed-form balanced-binary $S$, reference Fiedler vector
(for first-layer recovery), and reference leaf-cluster ids
(for final-partition ARI). All three constant across the $(p, \text{seed})$ sweep.

In [ ]:
refs: dict[int, dict] = {}
from src.utils.recursive_partition import recursive_split
for n in N_VALUES:
    full_key = make_full_key("balanced_binary", alpha=ALPHA, n=n)

    def builder(n=n, full_key=full_key):
        S = build_balanced_binary_S(n, ALPHA)
        fiedler_full = compute_fiedler_of_S(S)
        meta = {"tree_model": "balanced_binary", "alpha": ALPHA, "n": n,
                "dtype": str(S.dtype), "full_key": full_key}
        return S, fiedler_full, meta

    S, fiedler_full = get_or_compute_full(CACHE_DIR, full_key, builder)
    cluster_ids_M, bipartitions_M = recursive_split(
        S, threshold=THRESHOLD, num_gaps=NUM_GAPS, min_split=MIN_SPLIT,
    )
    refs[n] = {
        "S": S,
        "fiedler_full": fiedler_full,
        "cluster_ids_M": cluster_ids_M,
        "bipartitions_M": bipartitions_M,
        "n_clusters_M": int(cluster_ids_M.max() + 1),
        "n_bipartitions_M": len(bipartitions_M),
    }
    print(f"n={n:>5}  leaf clusters = {refs[n]['n_clusters_M']}, "
          f"bipartitions = {refs[n]['n_bipartitions_M']}")


## Cell 3 — Sweep: per $(n, p, \text{seed})$ compute both metrics
First-layer recovery uses `compute_recovery` (sign-agreement floored at $0.5$).
Final-partition ARI uses `compute_final_partition_agreement` with the cached
`cluster_ids_M` reference.

In [ ]:
rows = []
for n in N_VALUES:
    S = refs[n]["S"]
    fiedler_full = refs[n]["fiedler_full"]
    cluster_ids_M = refs[n]["cluster_ids_M"]
    bipartitions_M = refs[n]["bipartitions_M"]
    n_bp_M = len(bipartitions_M)
    for p in P_GRID:
        for seed in range(N_TRIALS):
            S_hat = subsample_S(S, float(p), int(seed))
            if float(p) * n < 1.0:
                recovery = 0.5
                ari = 0.0
                jaccard = 0.0
                n_S = -1
                shared = 0
            else:
                fiedler_hat = compute_fiedler_of_S(S_hat, sampling_prob=float(p))
                recovery = compute_recovery(fiedler_full, fiedler_hat)
                fp = compute_final_partition_agreement(
                    S, S_hat, threshold=THRESHOLD,
                    num_gaps=NUM_GAPS, min_split=MIN_SPLIT,
                    cluster_ids_M=cluster_ids_M,
                    bipartitions_M=bipartitions_M,
                )
                ari = fp["ari"]
                jaccard = fp["jaccard"]
                n_S = fp["n_clusters_S"]
                shared = fp["n_bipartitions_shared"]
            rows.append({"n": n, "p": float(p), "seed": seed,
                          "recovery": recovery, "ari": ari, "jaccard": jaccard,
                          "n_clusters_S": n_S, "n_bipartitions_M": n_bp_M,
                          "n_bipartitions_shared": shared})
    print(f"  n={n:>5}: {len(P_GRID) * N_TRIALS} trials done")

df = pd.DataFrame(rows)
df_agg = (df.groupby(["n", "p"], as_index=False)
            .agg(recovery=("recovery", "mean"),
                 ari=("ari", "mean"),
                 jaccard=("jaccard", "mean"),
                 jaccard_std=("jaccard", "std"),
                 ari_std=("ari", "std"),
                 n_clusters_S_mean=("n_clusters_S", "mean"),
                 shared_mean=("n_bipartitions_shared", "mean")))
df_agg.head()


## Cell 4 — Plots
Three square panels: first-layer sign-recovery (gray-to-red across $n$), bipartition Jaccard (headline), and ARI (for comparison).

In [ ]:
from matplotlib import cm

norm = plt.Normalize(vmin=min(N_VALUES), vmax=max(N_VALUES))
cmap = cm.get_cmap("YlOrRd")
colors = {n: cmap(0.35 + 0.6 * norm(n)) for n in N_VALUES}

fig, ax = plt.subplots(figsize=(7, 7))
for n in N_VALUES:
    sub = df_agg[df_agg["n"] == n].sort_values("p")
    ax.plot(sub["p"], sub["recovery"], marker="o", ms=4, lw=1.5,
            color=colors[n], label=f"n = {n}")
ax.set_xscale("log")
ax.set_xlabel("sub-sampling fraction $")
ax.set_ylabel("first-layer sign-recovery (mean over seeds)")
ax.set_ylim(0.45, 1.02)
ax.axhline(0.95, color="gray", lw=0.8, ls=":", label="0.95 floor")
ax.set_title(rf"First-layer recovery, balanced binary, $\alpha={ALPHA}$")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for n in N_VALUES:
    sub = df_agg[df_agg["n"] == n].sort_values("p")
    ax.plot(sub["p"], sub["jaccard"], marker="o", ms=4, lw=1.5,
            color=colors[n], label=f"n = {n}")
    ax.fill_between(sub["p"], sub["jaccard"] - sub["jaccard_std"],
                     sub["jaccard"] + sub["jaccard_std"],
                     color=colors[n], alpha=0.12, linewidth=0)
ax.set_xscale("log")
ax.set_xlabel("sub-sampling fraction $")
ax.set_ylabel("bipartition Jaccard (mean $\pm$ 1 std)")
ax.set_ylim(-0.02, 1.02)
ax.axhline(0.95, color="gray", lw=0.8, ls=":", label="0.95 floor")
ax.set_title(rf"Final-partition Jaccard ($\tau={THRESHOLD}$), balanced binary, $\alpha={ALPHA}$")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for n in N_VALUES:
    sub = df_agg[df_agg["n"] == n].sort_values("p")
    ax.plot(sub["p"], sub["ari"], marker="o", ms=4, lw=1.5,
            color=colors[n], label=f"n = {n}")
    ax.fill_between(sub["p"], sub["ari"] - sub["ari_std"], sub["ari"] + sub["ari_std"],
                     color=colors[n], alpha=0.12, linewidth=0)
ax.set_xscale("log")
ax.set_xlabel("sub-sampling fraction $")
ax.set_ylabel("final-partition ARI (mean $\pm$ 1 std)")
ax.set_ylim(-0.05, 1.05)
ax.axhline(0.95, color="gray", lw=0.8, ls=":", label="0.95 floor")
ax.set_title(rf"Final-partition ARI ($\tau={THRESHOLD}$), balanced binary, $\alpha={ALPHA}$ (for comparison)")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Cell 6 — Save aggregated tables

In [ ]:
out_dir = Path.cwd() / "stdr_partition_recovery_outputs"  # next to this notebook
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "trials.csv", index=False)
df_agg.to_csv(out_dir / "agg.csv", index=False)
print("Wrote:")
print("  ", out_dir / "trials.csv")
print("  ", out_dir / "agg.csv")
